In [11]:
import os
import sys
import platform
from lakehouse import bronze
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

In [12]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [13]:
builder = (
    SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
    .master("local[4]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [14]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up

In [15]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")

DataFrame[]

In [16]:
options = {
    "catalog": CATALOG,
    "target_schema": "bronze",
}

# 2 Stream


In [17]:
path = f"D:/{CATALOG}/streamdata/"
df_json_1 = spark.createDataFrame([(100, "Hyukjin Kwon1"),], ["age", "name"])
df_json_1.coalesce(1).write.mode("overwrite").format("json").save(path)


In [18]:
class TestStream(bronze.Bronze):
    def custom_load(self, table):
        df = spark.readStream.schema("age BIGINT, name STRING").json(path)
        return df
    
    def checkpoint_path(self, table):
        return f"{self.catalog}.{self.target_schema}.{table}/checkpoint"

instance = TestStream(spark, **options)

In [19]:
(
    instance.load()
    .transform()
    .write(mode="stream")
    .execute("stream")
)
spark.sql(f"SELECT * FROM {CATALOG}.bronze.stream").show(truncate=False)

2025-02-28 19:20:35 | stream | execute | Started
2025-02-28 19:20:35 | stream | load | Started
2025-02-28 19:20:35 | stream | load | Completed in 0.0 min
2025-02-28 19:20:35 | stream | transform | Started
2025-02-28 19:20:35 | stream | transform | Completed in 0.0 min
2025-02-28 19:20:35 | stream | write | Started
2025-02-28 19:21:00 | stream | write | Completed in 0.42 min
2025-02-28 19:21:00 | stream | execute | Completed in 0.42 min


+-----------------------+---+-------------+
|LH_BronzeTS            |age|name         |
+-----------------------+---+-------------+
|2025-02-28 19:20:43.069|100|Hyukjin Kwon1|
+-----------------------+---+-------------+



# 6 Clean Up

In [10]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")

DataFrame[]